In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [1]:
%cd /content
!git clone https://github.com/ammarhafiy/ds440-language-gap.git
%cd /content/ds440-language-gap
!pip install -q -r requirements.txt

/content
Cloning into 'ds440-language-gap'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (213/213), done.
remote: Total 222 (delta 89), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 935.45 KiB | 1.41 MiB/s, done.
Resolving deltas: 100% (89/89), done.
/content/ds440-language-gap


In [2]:
import os, requests
from google.colab import userdata
os.environ["SEALION_API_KEY"] = userdata.get("SEALION_API_KEY")
r = requests.get("https://api.sea-lion.ai/v1/models",
                 headers={"Authorization": f"Bearer {os.environ['SEALION_API_KEY']}"})
print(r.status_code)
for m in r.json().get("data", []):
    print(m["id"])

200
aisingapore/Llama-SEA-LION-v3-70B-IT
aisingapore/SEA-Guard
aisingapore/Gemma-SEA-LION-v4-27B-IT
aisingapore/Qwen-SEA-LION-v4.5-27B-IT
aisingapore/Qwen-SEA-LION-v4-32B-IT
BAAI/bge-m3
BAAI/bge-reranker-v2-m3
aisingapore/SEA-LION-E5-Embedding-600M
aisingapore/SEA-LION-ModernBERT-Embedding-300M
aisingapore/SEA-LION-ModernBERT-Embedding-600M
aisingapore/Nemotron-SEA-LION-v4.8-120B-A12B
aisingapore/Gemma-SEA-LION-Guard-v2-31B


In [3]:
import csv, io, os, subprocess, time
from google.colab import auth
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload, MediaIoBaseDownload

auth.authenticate_user()                      # one-time permission popup
drive_api = build("drive", "v3")

FILE_NAME = "sealion_Qwen-SEA-LION-v4.5-27B-IT.csv"
LOCAL = "/content/sealion_run.csv"

# Find the replies file in the ds440_responses folder on Google Drive
folders = drive_api.files().list(
    q="name = 'ds440_responses' and mimeType = 'application/vnd.google-apps.folder' and trashed = false",
    fields="files(id)").execute()["files"]
if len(folders) != 1:
    raise SystemExit(f"Expected 1 folder called ds440_responses, found {len(folders)}. Stop and tell Claude.")
found = drive_api.files().list(
    q=f"name = '{FILE_NAME}' and '{folders[0]['id']}' in parents and trashed = false",
    fields="files(id)").execute()["files"]
if len(found) != 1:
    raise SystemExit(f"Expected 1 file called {FILE_NAME}, found {len(found)}. Stop and tell Claude.")
FILE_ID = found[0]["id"]

def count_rows(path):
    with open(path, newline="", encoding="utf-8-sig") as f:
        return sum(1 for _ in csv.DictReader(f))

# Start from the copy on Google Drive (unless this Colab session already has a newer local copy)
if not os.path.exists(LOCAL):
    with io.FileIO(LOCAL, "wb") as fh:
        dl = MediaIoBaseDownload(fh, drive_api.files().get_media(fileId=FILE_ID))
        done = False
        while not done:
            _, done = dl.next_chunk()
START_ROWS = count_rows(LOCAL)
print(f"Starting from {START_ROWS} replies.")

def save_to_drive():
    """Upload the local file to Google Drive and wait until Drive confirms it."""
    rows = count_rows(LOCAL)
    if rows < START_ROWS:
        raise SystemExit("The local file has fewer replies than at the start; not uploading. Tell Claude.")
    for attempt in range(5):
        try:
            info = drive_api.files().update(
                fileId=FILE_ID, media_body=MediaFileUpload(LOCAL, mimetype="text/csv"),
                fields="modifiedTime").execute()
            return rows, info["modifiedTime"]
        except Exception as e:
            print("  upload failed, trying again:", e)
            time.sleep(30)
    raise SystemExit("Could not save to Google Drive after 5 tries. Tell Claude.")

args = ["python", "scripts/05_collect_responses.py", "--provider", "openai",
        "--base-url", "https://api.sea-lion.ai/v1", "--api-key-env", "SEALION_API_KEY",
        "--model", "aisingapore/Qwen-SEA-LION-v4.5-27B-IT", "--max-tokens", "256", "--sleep", "5",
        "--extra-body", '{"chat_template_kwargs": {"enable_thinking": false}}',
        "--out", LOCAL, "--limit", "200"]
for i in range(60):
    run = subprocess.run(args, capture_output=True, text=True)
    rows, when = save_to_drive()
    lines = run.stdout.splitlines()
    print(f"Batch {i+1}: {lines[0] if lines else ''} | SAVED to Drive: {rows} replies (Drive time {when})")
    if run.returncode != 0:
        print("Stopped with an error:", run.stderr[-500:])
        break
    if " 0 to send" in run.stdout:
        print("All prompts done.")
        break

Starting from 7554 replies.
  upload failed, trying again: [Errno 32] Broken pipe
Batch 1: 8390 prompts selected, 7534 already done, 200 to send (limited by --limit 200). | SAVED to Drive: 7754 replies (Drive time 2026-10-29T16:56:39.105Z)
  upload failed, trying again: [Errno 32] Broken pipe
Batch 2: 8390 prompts selected, 7734 already done, 200 to send (limited by --limit 200). | SAVED to Drive: 7954 replies (Drive time 2026-10-29T17:19:06.528Z)


  upload failed, trying again: [Errno 32] Broken pipe
Batch 3: 8390 prompts selected, 7934 already done, 200 to send (limited by --limit 200). | SAVED to Drive: 8154 replies (Drive time 2026-10-29T17:41:13.985Z)
  upload failed, trying again: [Errno 32] Broken pipe
Batch 4: 8390 prompts selected, 8134 already done, 200 to send (limited by --limit 200). | SAVED to Drive: 8354 replies (Drive time 2026-10-29T18:03:34.449Z)
  upload failed, trying again: [Errno 32] Broken pipe
Batch 5: 8390 prompts selected, 8334 already done, 56 to send (limited by --limit 200). | SAVED to Drive: 8410 replies (Drive time 2026-10-29T18:10:15.472Z)
Batch 6: 8390 prompts selected, 8390 already done, 0 to send (limited by --limit 200). | SAVED to Drive: 8410 replies (Drive time 2026-10-29T18:10:18.650Z)
All prompts done.


In [9]:
import pandas as pd
df = pd.read_csv("/content/drive/MyDrive/ds440_responses/sealion_Qwen-SEA-LION-v4.5-27B-IT.csv")
print(len(df), "replies,", (df["error"].fillna("") != "").sum(), "errors")
print(df["finish_reason"].value_counts())
df[["prompt_id", "response"]].head(20)

1137 replies, 0 errors
finish_reason
stop    1137
Name: count, dtype: int64


,prompt_id,response
0,msl-Q16231365-P1-en-no_facts,"Kuala Lumpur, Malaysia"
1,msl-Q16231365-P1-en-with_facts,Kempas
2,msl-Q16231365-P1-ms-no_facts,Kamarul Afiq Kamaruddin dilahirkan di **Kuala ...
3,msl-Q16231365-P1-ms-with_facts,Kempas
4,msl-Q16231365-P2-en-no_facts,2000
5,msl-Q16231365-P2-en-with_facts,1986
6,msl-Q16231365-P2-ms-no_facts,Kamarul Afiq Kamaruddin dilahirkan pada tahun ...
7,msl-Q16231365-P2-ms-with_facts,1986
8,msl-Q6893888-P1-en-no_facts,"Kuala Lumpur, Malaysia"
9,msl-Q6893888-P1-en-with_facts,Selangor


In [ ]:
import os, time, requests
KEY = os.environ["SEALION_API_KEY"]
URL = "https://api.sea-lion.ai/v1/chat/completions"
prompt = ("Answer with a short answer only, in the same language as the question.\n\n"
          "Question: Where was the footballer Kamarul Afiq Kamaruddin born?")
tests = [("aisingapore/Qwen-SEA-LION-v4.5-27B-IT", {"thinking_mode": "off"}),
         ("aisingapore/Qwen-SEA-LION-v4.5-27B-IT", {"enable_thinking": False}),
         ("aisingapore/Gemma-SEA-LION-v4-27B-IT", None)]
for model, kw in tests:
    body = {"model": model, "messages": [{"role": "user", "content": prompt}],
            "max_completion_tokens": 256, "temperature": 0}
    if kw: body["chat_template_kwargs"] = kw
    r = requests.post(URL, headers={"Authorization": f"Bearer {KEY}"}, json=body, timeout=120)
    print("==", model, kw, "| HTTP", r.status_code)
    if r.ok:
        c = r.json()["choices"][0]
        print("finish:", c.get("finish_reason"), "| tokens:", r.json().get("usage"))
        print("answer:", repr((c["message"].get("content") or "")[:150]))
    else:
        print(r.text[:300])
    time.sleep(7)

== aisingapore/Qwen-SEA-LION-v4.5-27B-IT {'thinking_mode': 'off'} | HTTP 200
finish: length | tokens: {'completion_tokens': 256, 'prompt_tokens': 42, 'total_tokens': 298, 'completion_tokens_details': {'reasoning_tokens': 256}}
answer: ''
== aisingapore/Qwen-SEA-LION-v4.5-27B-IT {'enable_thinking': False} | HTTP 200
finish: stop | tokens: {'completion_tokens': 6, 'prompt_tokens': 44, 'total_tokens': 50, 'completion_tokens_details': {'reasoning_tokens': 0}}
answer: 'Kuala Lumpur, Malaysia'
== aisingapore/Gemma-SEA-LION-v4-27B-IT None | HTTP 200
finish: stop | tokens: {'completion_tokens': 5, 'prompt_tokens': 40, 'total_tokens': 45}
answer: 'Kuala Lumpur.'


In [ ]:
!rm "/content/drive/MyDrive/ds440_responses/sealion_Qwen-SEA-LION-v4.5-27B-IT.csv"
%cd /content/ds440-language-gap
!git pull

rm: cannot remove '/content/drive/MyDrive/ds440_responses/sealion_Qwen-SEA-LION-v4.5-27B-IT.csv': No such file or directory
/content/ds440-language-gap
Already up to date.


Drive saved and disconnected.


In [4]:
%cd /content/ds440-language-gap
import csv
rows = list(csv.DictReader(open("/content/sealion_run.csv", encoding="utf-8-sig")))
prompts = {r["prompt_id"] for r in csv.DictReader(open("data/prompts/prompts.csv", encoding="utf-8-sig"))}
ids = [r["prompt_id"] for r in rows]
print("Rows in file:", len(rows))
print("Duplicate prompt IDs:", len(ids) - len(set(ids)))
print("Prompts with a reply:", len(prompts & set(ids)), "of", len(prompts))
print("Replies to removed questions (left out when scoring):", len(set(ids) - prompts))
print("Errors:", sum(1 for r in rows if r["error"]))
print("Empty replies:", sum(1 for r in rows if not r["response"].strip()))
print("Cut off at the length limit:", sum(1 for r in rows if r["finish_reason"].lower() in ("length", "max_tokens")))
print("Model returned:", sorted({r["model_returned"] for r in rows}))
print("First / last reply:", min(r["timestamp_utc"] for r in rows), "/", max(r["timestamp_utc"] for r in rows))

!mkdir -p data/responses
!cp /content/sealion_run.csv data/responses/sealion_Qwen-SEA-LION-v4.5-27B-IT.csv
!python scripts/06_score.py --files data/responses/sealion_Qwen-SEA-LION-v4.5-27B-IT.csv

/content/ds440-language-gap
Rows in file: 8410
Duplicate prompt IDs: 0
Prompts with a reply: 8390 of 8390
Replies to removed questions (left out when scoring): 20
Errors: 0
Empty replies: 0
Cut off at the length limit: 0
Model returned: ['aisingapore/Qwen-SEA-LION-v4.5-27B-IT']
First / last reply: 2026-10-27T18:49:06+00:00 / 2026-10-29T18:09:39+00:00

sealion_Qwen-SEA-LION-v4.5-27B-IT.csv: 8390 replies scored, 0 failed requests skipped
  20 replies to 5 questions no longer in the question set were left out
  source    league lang condition    correct  accuracy  95% CI
  mkqa      mixed  en   no_facts     62/79    0.79      [0.68, 0.86]
  mkqa      mixed  ms   no_facts     57/79    0.72      [0.61, 0.81]
  wikidata  msl    en   no_facts    128/1003  0.13      [0.11, 0.15]
  wikidata  msl    en   with_facts 1002/1003  1.00      [0.99, 1.00]
  wikidata  msl    ms   no_facts    120/1003  0.12      [0.10, 0.14]
  wikidata  msl    ms   with_facts 1003/1003  1.00      [1.00, 1.00]
  wikidata 